In [0]:
import gzip
import io
import shlex
import uuid
from datetime import UTC, datetime

import requests
from pyspark.sql import Row

In [0]:
dbutils.widgets.text("trigger_type", "manual")
dbutils.widgets.text("run_user", "anna")

trigger_type = dbutils.widgets.get("trigger_type")
run_user = dbutils.widgets.get("run_user")

In [0]:
storage_account = "annagaplanyandatabricks"
container = "pdb-data"

spark.conf.set(
    f"fs.azure.account.key.{storage_account}.dfs.core.windows.net",
    dbutils.secrets.get(scope="adls-scope", key="storage-key")
)

base_path = f"abfss://{container}@{storage_account}.dfs.core.windows.net/pdb_lab"
bronze_path = f"{base_path}/bronze"
system_path = f"{base_path}/system"

print(f"Base path: {base_path}")

In [0]:
spark.sql("CREATE DATABASE IF NOT EXISTS pdb_lab")

In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS pdb_lab.pipeline_logs (
    log_id STRING,
    run_id STRING,
    stage STRING,
    table_name STRING,
    status STRING,
    record_count BIGINT,
    message STRING,
    created_at TIMESTAMP
) USING DELTA
LOCATION '{system_path}/pipeline_logs'
""")

In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS pdb_lab.config_pdb_actualizer(
    pdb_id STRING,
    is_active BOOLEAN,
    requested_by STRING,
    requested_at TIMESTAMP
) USING DELTA
LOCATION '{system_path}/config_pdb_actualizer'
""")

In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS pdb_lab.history_pdb_actualizer(
    run_id STRING,
    run_user STRING,
    started_at TIMESTAMP,
    finished_at TIMESTAMP,
    status STRING,
    trigger_type STRING,
    input_ids ARRAY<STRING>,
    error_message STRING
) USING DELTA
LOCATION '{system_path}/history_pdb_actualizer'
""")

In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS pdb_lab.register_pdb_actualizer(
    pdb_id STRING,
    last_actualized_at TIMESTAMP,
    status STRING,
    source_url STRING
) USING DELTA
LOCATION '{system_path}/register_pdb_actualizer'
""")

In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS pdb_lab.bronze_entity(
    run_id STRING,
    pdb_code STRING,
    source_url STRING,
    ingested_at TIMESTAMP,
    entity_id STRING,
    details STRING,
    formula_weight STRING,
    pdbx_description STRING,
    pdbx_ec STRING,
    pdbx_fragment STRING,
    pdbx_mutation STRING,
    pdbx_number_of_molecules STRING,
    src_method STRING,
    entity_type STRING
) USING DELTA
LOCATION '{bronze_path}/bronze_entity'
""")

In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS pdb_lab.bronze_exptl (
    run_id STRING,
    pdb_code STRING,
    source_url STRING,
    ingested_at TIMESTAMP,
    entry_id STRING,
    method STRING
) USING DELTA
LOCATION '{bronze_path}/bronze_exptl'
""")

In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS pdb_lab.bronze_pdbx_database_pdb_obs_spr (
    run_id STRING,
    pdb_code STRING,
    source_url STRING,
    ingested_at TIMESTAMP,
    obs_pdb_id STRING,
    replace_pdb_id STRING,
    obs_date STRING,
    details STRING,
    obs_id STRING
) USING DELTA
LOCATION '{bronze_path}/bronze_pdbx_database_pdb_obs_spr'
""")

In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS pdb_lab.bronze_entity_poly_seq (
    run_id STRING,
    pdb_code STRING,
    source_url STRING,
    ingested_at TIMESTAMP,
    entity_id STRING,
    mon_id STRING,
    seq_num STRING,
    hetero STRING
) USING DELTA
LOCATION '{bronze_path}/bronze_entity_poly_seq'
""")

In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS pdb_lab.bronze_chem_comp (
    run_id STRING,
    pdb_code STRING,
    source_url STRING,
    ingested_at TIMESTAMP,
    chem_comp_id STRING,
    formula STRING,
    formula_weight STRING,
    model_details STRING,
    model_erf STRING,
    model_source STRING,
    mon_nstd_class STRING,
    mon_nstd_details STRING,
    mon_nstd_flag STRING,
    mon_nstd_parent STRING,
    mon_nstd_parent_comp_id STRING,
    name STRING,
    number_atoms_all STRING,
    number_atoms_nh STRING,
    one_letter_code STRING,
    pdbx_ambiguous_flag STRING,
    pdbx_casnum STRING,
    pdbx_class_1 STRING,
    pdbx_class_2 STRING,
    pdbx_comp_type STRING,
    pdbx_component_no STRING,
    pdbx_formal_charge STRING,
    pdbx_ideal_coordinates_details STRING,
    pdbx_ideal_coordinates_missing_flag STRING,
    pdbx_initial_date STRING,
    pdbx_model_coordinates_db_code STRING,
    pdbx_model_coordinates_details STRING,
    pdbx_model_coordinates_missing_flag STRING,
    pdbx_modification_details STRING,
    pdbx_modified_date STRING,
    pdbx_nscnum STRING,
    pdbx_number_subcomponents STRING,
    pdbx_pcm STRING,
    pdbx_processing_site STRING,
    pdbx_release_status STRING,
    pdbx_replaced_by STRING,
    pdbx_replaces STRING,
    pdbx_reserved_name STRING,
    pdbx_smiles STRING,
    pdbx_status STRING,
    pdbx_subcomponent_list STRING,
    pdbx_synonyms STRING,
    pdbx_type STRING,
    pdbx_type_modified STRING,
    three_letter_code STRING,
    chem_comp_type STRING
) USING DELTA
LOCATION '{bronze_path}/bronze_chem_comp'
""")

In [0]:
def start_run(trigger_type: str, pdb_ids: list[str], run_user: str = "anna") -> str:
    run_id = str(uuid.uuid4())
    ids_sql = ", ".join([f"'{x}'" for x in pdb_ids])

    spark.sql(f"""
        INSERT INTO pdb_lab.history_pdb_actualizer
        SELECT
            '{run_id}' AS run_id,
            '{run_user}' AS run_user,
            current_timestamp() AS started_at,
            CAST(NULL AS TIMESTAMP) AS finished_at,
            'RUNNING' AS status,
            '{trigger_type}' AS trigger_type,
            array({ids_sql}) AS input_ids,
            CAST(NULL AS STRING) AS error_message
    """)
    return run_id


In [0]:
def finish_run_success(run_id: str) -> None:
    spark.sql(f"""
        UPDATE pdb_lab.history_pdb_actualizer
        SET finished_at = current_timestamp(),
            status = 'SUCCESS'
        WHERE run_id = '{run_id}'
    """)

In [0]:
def finish_run_failed(run_id: str, error_message: str) -> None:
    err = error_message.replace("'", "''")
    spark.sql(f"""
        UPDATE pdb_lab.history_pdb_actualizer
        SET finished_at = current_timestamp(),
            status = 'FAILED',
            error_message = '{err}'
        WHERE run_id = '{run_id}'
    """)

In [0]:
def log_event(run_id, stage, table_name, status, record_count=None, message=None):
    log_id = str(uuid.uuid4())
    msg = (message or "").replace("'", "''")

    spark.sql(f"""
        INSERT INTO pdb_lab.pipeline_logs
        SELECT
            '{log_id}',
            '{run_id}',
            '{stage}',
            '{table_name}',
            '{status}',
            {record_count if record_count is not None else "NULL"},
            '{msg}',
            current_timestamp()
    """)

In [0]:
def update_register(pdb_id: str, status: str) -> None:
    source_url = f"https://files.wwpdb.org/pub/pdb/data/structures/all/mmCIF/{pdb_id}.cif.gz"

    spark.sql(f"""
        MERGE INTO pdb_lab.register_pdb_actualizer t
        USING (
            SELECT
                '{pdb_id}' AS pdb_id,
                current_timestamp() AS last_actualized_at,
                '{status}' AS status,
                '{source_url}' AS source_url
        ) s
        ON t.pdb_id = s.pdb_id
        WHEN MATCHED THEN UPDATE SET
            t.last_actualized_at = s.last_actualized_at,
            t.status = s.status,
            t.source_url = s.source_url
        WHEN NOT MATCHED THEN INSERT *
    """)

In [0]:
def get_active_pdb_ids() -> list[str]:
    pdb_ids = [
        str(r["pdb_id"]).lower().strip()
        for r in spark.sql("""
            SELECT DISTINCT pdb_id
            FROM pdb_lab.config_pdb_actualizer
            WHERE is_active = true
        """).collect()
    ]

    if not pdb_ids:
        raise ValueError("No active pdb_ids found")

    for pdb_id in pdb_ids:
        if len(pdb_id) != 4 or not pdb_id.isalnum():
            raise ValueError(f"Invalid pdb_id: {pdb_id}")

    return pdb_ids

In [0]:
def download_mmcif_text(pdb_id: str) -> str:
    url = f"https://files.wwpdb.org/pub/pdb/data/structures/all/mmCIF/{pdb_id}.cif.gz"
    response = requests.get(url, timeout=120)
    response.raise_for_status()

    with gzip.GzipFile(fileobj=io.BytesIO(response.content)) as gz:
        return gz.read().decode("utf-8", errors="replace")

In [0]:
def clean_cif_value(v):
    if v is None:
        return None

    v = str(v).strip()

    if v in {"?", "."}:
        return None

    if len(v) >= 2 and ((v[0] == "'" and v[-1] == "'") or (v[0] == '"' and v[-1] == '"')):
        v = v[1:-1]

    return v

In [0]:
def split_cif_lines(text: str) -> list[str]:
    return text.splitlines()

In [0]:
def tokenize_cif_row(row: str) -> list[str]:
    return shlex.split(row, posix=True)

In [0]:
def extract_single_value(lines: list[str], item_name: str):
    for i, line in enumerate(lines):
        stripped = line.strip()

        if stripped.startswith(item_name + " "):
            return stripped[len(item_name):].strip()

        if stripped == item_name and i + 1 < len(lines):
            return lines[i + 1].strip()

    return None

In [0]:
def parse_loop_blocks(lines: list[str]) -> list[dict]:
    loops = []
    i = 0
    n = len(lines)

    while i < n:
        line = lines[i].strip()

        if line == "loop_":
            i += 1
            cols = []

            while i < n and lines[i].strip().startswith("_"):
                cols.append(lines[i].strip())
                i += 1

            rows = []
            current_tokens = []

            while i < n:
                raw = lines[i]
                stripped = raw.strip()

                if stripped == "" or stripped == "#":
                    if current_tokens:
                        if len(current_tokens) >= len(cols):
                            rows.append(current_tokens[:len(cols)])
                        current_tokens = []
                    i += 1
                    if stripped == "#":
                        break
                    continue

                if stripped == "loop_" or stripped.startswith("_"):
                    if current_tokens:
                        if len(current_tokens) >= len(cols):
                            rows.append(current_tokens[:len(cols)])
                        current_tokens = []
                    break

                if stripped.startswith(";"):
                    multi = [raw[1:]]
                    i += 1
                    while i < n and not lines[i].startswith(";"):
                        multi.append(lines[i])
                        i += 1
                    current_tokens.append("\n".join(multi).strip())
                    if i < n:
                        i += 1
                    continue

                current_tokens.extend(tokenize_cif_row(raw))

                while len(current_tokens) >= len(cols):
                    rows.append(current_tokens[:len(cols)])
                    current_tokens = current_tokens[len(cols):]

                i += 1

            loops.append({"columns": cols, "rows": rows})
            continue

        i += 1

    return loops

In [0]:
def extract_loop_category(loops: list[dict], prefix: str) -> list[dict]:
    out = []

    for loop in loops:
        cols = loop["columns"]
        if cols and all(c.startswith(prefix) for c in cols):
            short_cols = [c[len(prefix):] for c in cols]
            for row in loop["rows"]:
                item = {}
                for c, v in zip(short_cols, row):
                    item[c] = v
                out.append(item)

    return out

In [0]:
def extract_exptl(lines: list[str], loops: list[dict]) -> list[dict]:
    rows = extract_loop_category(loops, "_exptl.")
    if rows:
        return rows

    entry_id = extract_single_value(lines, "_exptl.entry_id")
    method = extract_single_value(lines, "_exptl.method")

    if entry_id is not None or method is not None:
        return [{"entry_id": entry_id, "method": method}]

    return []

In [0]:
def process_pdb_file_full(pdb_id: str, run_id: str) -> dict:
    source_url = f"https://files.wwpdb.org/pub/pdb/data/structures/all/mmCIF/{pdb_id}.cif.gz"
    ingested_at = datetime.now(UTC)

    text = download_mmcif_text(pdb_id)
    lines = split_cif_lines(text)
    loops = parse_loop_blocks(lines)

    entity_rows = extract_loop_category(loops, "_entity.")
    exptl_rows = extract_exptl(lines, loops)
    obs_rows = extract_loop_category(loops, "_pdbx_database_PDB_obs_spr.")
    entity_poly_seq_rows = extract_loop_category(loops, "_entity_poly_seq.")
    chem_comp_rows = extract_loop_category(loops, "_chem_comp.")

    entity_out = [
        Row(
            run_id=run_id,
            pdb_code=pdb_id,
            source_url=source_url,
            ingested_at=ingested_at,
            entity_id=clean_cif_value(r.get("id")),
            details=clean_cif_value(r.get("details")),
            formula_weight=clean_cif_value(r.get("formula_weight")),
            pdbx_description=clean_cif_value(r.get("pdbx_description")),
            pdbx_ec=clean_cif_value(r.get("pdbx_ec")),
            pdbx_fragment=clean_cif_value(r.get("pdbx_fragment")),
            pdbx_mutation=clean_cif_value(r.get("pdbx_mutation")),
            pdbx_number_of_molecules=clean_cif_value(r.get("pdbx_number_of_molecules")),
            src_method=clean_cif_value(r.get("src_method")),
            entity_type=clean_cif_value(r.get("type")),
        )
        for r in entity_rows
    ]

    exptl_out = [
        Row(
            run_id=run_id,
            pdb_code=pdb_id,
            source_url=source_url,
            ingested_at=ingested_at,
            entry_id=clean_cif_value(r.get("entry_id")),
            method=clean_cif_value(r.get("method")),
        )
        for r in exptl_rows
    ]

    obs_out = [
        Row(
            run_id=run_id,
            pdb_code=pdb_id,
            source_url=source_url,
            ingested_at=ingested_at,
            obs_pdb_id=clean_cif_value(r.get("pdb_id")),
            replace_pdb_id=clean_cif_value(r.get("replace_pdb_id")),
            obs_date=clean_cif_value(r.get("date")),
            details=clean_cif_value(r.get("details")),
            obs_id=clean_cif_value(r.get("id")),
        )
        for r in obs_rows
    ]

    entity_poly_seq_out = [
        Row(
            run_id=run_id,
            pdb_code=pdb_id,
            source_url=source_url,
            ingested_at=ingested_at,
            entity_id=clean_cif_value(r.get("entity_id")),
            mon_id=clean_cif_value(r.get("mon_id")),
            seq_num=clean_cif_value(r.get("num")),
            hetero=clean_cif_value(r.get("hetero")),
        )
        for r in entity_poly_seq_rows
    ]

    chem_comp_out = [
        Row(
            run_id=run_id,
            pdb_code=pdb_id,
            source_url=source_url,
            ingested_at=ingested_at,
            chem_comp_id=clean_cif_value(r.get("id")),
            formula=clean_cif_value(r.get("formula")),
            formula_weight=clean_cif_value(r.get("formula_weight")),
            model_details=clean_cif_value(r.get("model_details")),
            model_erf=clean_cif_value(r.get("model_erf")),
            model_source=clean_cif_value(r.get("model_source")),
            mon_nstd_class=clean_cif_value(r.get("mon_nstd_class")),
            mon_nstd_details=clean_cif_value(r.get("mon_nstd_details")),
            mon_nstd_flag=clean_cif_value(r.get("mon_nstd_flag")),
            mon_nstd_parent=clean_cif_value(r.get("mon_nstd_parent")),
            mon_nstd_parent_comp_id=clean_cif_value(r.get("mon_nstd_parent_comp_id")),
            name=clean_cif_value(r.get("name")),
            number_atoms_all=clean_cif_value(r.get("number_atoms_all")),
            number_atoms_nh=clean_cif_value(r.get("number_atoms_nh")),
            one_letter_code=clean_cif_value(r.get("one_letter_code")),
            pdbx_ambiguous_flag=clean_cif_value(r.get("pdbx_ambiguous_flag")),
            pdbx_casnum=clean_cif_value(r.get("pdbx_casnum")),
            pdbx_class_1=clean_cif_value(r.get("pdbx_class_1")),
            pdbx_class_2=clean_cif_value(r.get("pdbx_class_2")),
            pdbx_comp_type=clean_cif_value(r.get("pdbx_comp_type")),
            pdbx_component_no=clean_cif_value(r.get("pdbx_component_no")),
            pdbx_formal_charge=clean_cif_value(r.get("pdbx_formal_charge")),
            pdbx_ideal_coordinates_details=clean_cif_value(r.get("pdbx_ideal_coordinates_details")),
            pdbx_ideal_coordinates_missing_flag=clean_cif_value(r.get("pdbx_ideal_coordinates_missing_flag")),
            pdbx_initial_date=clean_cif_value(r.get("pdbx_initial_date")),
            pdbx_model_coordinates_db_code=clean_cif_value(r.get("pdbx_model_coordinates_db_code")),
            pdbx_model_coordinates_details=clean_cif_value(r.get("pdbx_model_coordinates_details")),
            pdbx_model_coordinates_missing_flag=clean_cif_value(r.get("pdbx_model_coordinates_missing_flag")),
            pdbx_modification_details=clean_cif_value(r.get("pdbx_modification_details")),
            pdbx_modified_date=clean_cif_value(r.get("pdbx_modified_date")),
            pdbx_nscnum=clean_cif_value(r.get("pdbx_nscnum")),
            pdbx_number_subcomponents=clean_cif_value(r.get("pdbx_number_subcomponents")),
            pdbx_pcm=clean_cif_value(r.get("pdbx_pcm")),
            pdbx_processing_site=clean_cif_value(r.get("pdbx_processing_site")),
            pdbx_release_status=clean_cif_value(r.get("pdbx_release_status")),
            pdbx_replaced_by=clean_cif_value(r.get("pdbx_replaced_by")),
            pdbx_replaces=clean_cif_value(r.get("pdbx_replaces")),
            pdbx_reserved_name=clean_cif_value(r.get("pdbx_reserved_name")),
            pdbx_smiles=clean_cif_value(r.get("pdbx_smiles")),
            pdbx_status=clean_cif_value(r.get("pdbx_status")),
            pdbx_subcomponent_list=clean_cif_value(r.get("pdbx_subcomponent_list")),
            pdbx_synonyms=clean_cif_value(r.get("pdbx_synonyms")),
            pdbx_type=clean_cif_value(r.get("pdbx_type")),
            pdbx_type_modified=clean_cif_value(r.get("pdbx_type_modified")),
            three_letter_code=clean_cif_value(r.get("three_letter_code")),
            chem_comp_type=clean_cif_value(r.get("type")),
        )
        for r in chem_comp_rows
    ]

    return {
        "entity": entity_out,
        "exptl": exptl_out,
        "obs": obs_out,
        "entity_poly_seq": entity_poly_seq_out,
        "chem_comp": chem_comp_out,
    }


In [0]:
def append_rows_to_table(rows, table_name: str) -> None:
    if not rows:
        print(f"No rows to write for {table_name}")
        return

    target_schema = spark.table(table_name).schema
    df = spark.createDataFrame(rows, schema=target_schema)
    df.write.mode("append").saveAsTable(table_name)
    print(f"Wrote {len(rows)} rows to {table_name}")

In [0]:
def run_bronze_pipeline(trigger_type: str = "manual", run_user: str = "anna") -> str:
    pdb_ids = get_active_pdb_ids()
    run_id = start_run(trigger_type, pdb_ids, run_user)

    all_entity = []
    all_exptl = []
    all_obs = []
    all_entity_poly_seq = []
    all_chem_comp = []
    parsed_pdb_ids = []

    try:
        for pdb_id in pdb_ids:
            try:
                log_event(run_id, "bronze", f"pdb:{pdb_id}", "STARTED")

                result = process_pdb_file_full(pdb_id, run_id)

                all_entity.extend(result["entity"])
                all_exptl.extend(result["exptl"])
                all_obs.extend(result["obs"])
                all_entity_poly_seq.extend(result["entity_poly_seq"])
                all_chem_comp.extend(result["chem_comp"])

                parsed_pdb_ids.append(pdb_id)

                log_event(run_id, "bronze", f"pdb:{pdb_id}", "SUCCESS")

            except Exception as e:
                log_event(run_id, "bronze", f"pdb:{pdb_id}", "FAILED", message=str(e))
                update_register(pdb_id, "FAILED")
                raise ValueError(f"Failed processing pdb_id {pdb_id}: {str(e)}") from e

        tables_data = [
            ("pdb_lab.bronze_entity", all_entity),
            ("pdb_lab.bronze_exptl", all_exptl),
            ("pdb_lab.bronze_pdbx_database_pdb_obs_spr", all_obs),
            ("pdb_lab.bronze_entity_poly_seq", all_entity_poly_seq),
            ("pdb_lab.bronze_chem_comp", all_chem_comp),
        ]

        for table_name, rows in tables_data:
            try:
                log_event(run_id, "bronze", table_name, "STARTED")

                append_rows_to_table(rows, table_name)

                count = spark.table(table_name).count()

                log_event(run_id, "bronze", table_name, "SUCCESS", record_count=count)

            except Exception as e:
                log_event(run_id, "bronze", table_name, "FAILED", message=str(e))
                raise

        for pdb_id in parsed_pdb_ids:
            update_register(pdb_id, "SUCCESS")

        finish_run_success(run_id)
        return run_id

    except Exception as e:
        finish_run_failed(run_id, str(e))
        raise

In [0]:
run_id = run_bronze_pipeline(trigger_type=trigger_type, run_user=run_user)
print(f"Bronze pipeline finished. run_id = {run_id}")